# NYXOR · free GPU video studio (Google Colab)

This notebook runs an open text-to-video model on a **free Colab GPU** and exposes it to your PC through a temporary public URL.

1. **Runtime → Change runtime type → T4 GPU** (free tier is fine).
2. Run the three cells below, top to bottom (the first install takes a few minutes; the model download is ~10 GB).
3. The last cell prints a **connect string**. Paste it into NYXOR → VIDEO STUDIO (or into the quick-add box on the PROVIDERS tab).
4. Keep this tab open while you generate. A 6-second clip takes roughly 5–15 minutes on a free T4.

Security: the URL is random and every request needs the token inside the connect string. Anyone who has the string can use your GPU session, so don't share it. Stop the cell (or the runtime) when you're done.

In [ ]:
#@title 1 · Install
!pip -q install -U diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg
print("installed")

In [ ]:
#@title 2 · Load the video model
import torch
from diffusers import CogVideoXPipeline
from diffusers.utils import export_to_video

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
MODEL = "THUDM/CogVideoX-2b"  #@param ["THUDM/CogVideoX-2b"]
pipe = CogVideoXPipeline.from_pretrained(MODEL, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()   # keeps VRAM use low enough for the free 16 GB T4
pipe.vae.enable_tiling()
pipe.vae.enable_slicing()
print("loaded", MODEL, "on", torch.cuda.get_device_name(0))

In [ ]:
#@title 3 · Start the server and public tunnel
import json, os, queue, re, secrets, subprocess, threading, time, urllib.request, uuid
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

TOKEN = secrets.token_urlsafe(18)
OUT = "/content/nyxor_videos"; os.makedirs(OUT, exist_ok=True)
JOBS, Q = {}, queue.Queue()

def generate(job):
    gen = torch.Generator(device="cpu").manual_seed(job["seed"])
    def on_step(p, step, t, kw):
        job["progress"] = min(0.99, (step + 1) / job["steps"])
        if job["status"] == "cancelled":
            raise RuntimeError("cancelled")
        return kw
    frames = pipe(prompt=job["prompt"], num_videos_per_prompt=1, num_inference_steps=job["steps"],
                  num_frames=job["frames"], guidance_scale=6, generator=gen,
                  callback_on_step_end=on_step).frames[0]
    export_to_video(frames, os.path.join(OUT, job["id"] + ".mp4"), fps=job["fps"])

def worker():
    while True:
        job = JOBS[Q.get()]
        if job["status"] == "cancelled":
            continue
        job["status"] = "running"
        try:
            generate(job)
            if job["status"] != "cancelled":
                job["status"], job["progress"] = "done", 1.0
        except Exception as e:
            if job["status"] != "cancelled":
                job["status"], job["error"] = "error", str(e)[:300]
        finally:
            torch.cuda.empty_cache()
threading.Thread(target=worker, daemon=True).start()

def clamp(v, lo, hi, d):
    try: return max(lo, min(int(v), hi))
    except (TypeError, ValueError): return d

class H(BaseHTTPRequestHandler):
    def log_message(self, *a): pass
    def _send(self, code, body, ctype="application/json"):
        data = body if isinstance(body, bytes) else json.dumps(body).encode()
        self.send_response(code); self.send_header("Content-Type", ctype)
        self.send_header("Content-Length", str(len(data))); self.end_headers(); self.wfile.write(data)
    def _ok(self): return self.headers.get("Authorization") == "Bearer " + TOKEN
    def do_GET(self):
        if not self._ok(): return self._send(401, {"error": "bad token"})
        if self.path == "/health":
            return self._send(200, {"ok": True, "model": MODEL, "gpu": torch.cuda.get_device_name(0)})
        m = re.match(r"^/job/(\w+)$", self.path)
        if m and m.group(1) in JOBS:
            j = JOBS[m.group(1)]
            return self._send(200, {k: j[k] for k in ("status", "progress", "error")})
        m = re.match(r"^/video/(\w+)\.mp4$", self.path)
        if m and os.path.exists(os.path.join(OUT, m.group(1) + ".mp4")):
            return self._send(200, open(os.path.join(OUT, m.group(1) + ".mp4"), "rb").read(), "video/mp4")
        self._send(404, {"error": "not found"})
    def do_POST(self):
        if not self._ok(): return self._send(401, {"error": "bad token"})
        body = json.loads(self.rfile.read(int(self.headers.get("Content-Length", 0))) or b"{}")
        if self.path == "/generate":
            prompt = str(body.get("prompt", "")).strip()[:1000]
            if not prompt: return self._send(400, {"error": "empty prompt"})
            jid = uuid.uuid4().hex[:10]
            frames = clamp(body.get("frames"), 9, 49, 49); frames = frames - (frames - 1) % 8
            JOBS[jid] = dict(id=jid, prompt=prompt, status="queued", progress=0.0, error="",
                             steps=clamp(body.get("steps"), 10, 60, 40), frames=frames,
                             fps=clamp(body.get("fps"), 4, 24, 8), seed=clamp(body.get("seed"), 0, 2**31 - 1, int(time.time()) % 100000))
            Q.put(jid); return self._send(200, {"job": jid})
        m = re.match(r"^/cancel/(\w+)$", self.path)
        if m and m.group(1) in JOBS:
            JOBS[m.group(1)]["status"] = "cancelled"; return self._send(200, {"ok": True})
        self._send(404, {"error": "not found"})

srv = ThreadingHTTPServer(("127.0.0.1", 8000), H)
threading.Thread(target=srv.serve_forever, daemon=True).start()

urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "cloudflared")
os.chmod("cloudflared", 0o755)
tun = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL = None
for line in tun.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m: URL = m.group(0); break
threading.Thread(target=lambda: [None for _ in tun.stdout], daemon=True).start()
print("=" * 70)
print("PASTE THIS INTO NYXOR  (VIDEO STUDIO > Colab)")
print()
print(f"{URL}?token={TOKEN}")
print("=" * 70)
while True:
    time.sleep(60)